# MedGemma ACI Fine-tune — Inference Notebook
Loads the fine-tuned LoRA adapters from HuggingFace and runs evaluation on the ACI-Bench test set.
No training code, no Trainer — pure inference only.

In [1]:
!pip install -q bitsandbytes transformers peft datasets evaluate bert-score rouge_score huggingface_hub

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00


In [2]:
import math
import re
from functools import partial
from typing import Any

import evaluate
import numpy as np
import torch
from datasets import concatenate_datasets, load_dataset
from huggingface_hub import login
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [3]:
# ── Authentication ────────────────────────────────────────────────────────────
# Uncomment whichever secret source matches your environment.

# Google Colab:
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN_WRITE")

# Kaggle:
# from kaggle_secrets import UserSecretsClient
# hf_token = UserSecretsClient().get_secret("HF_TOKEN")

# Plain string (not recommended for shared notebooks):
# hf_token = "hf_..."

if hf_token:
    login(token=hf_token)
else:
    raise ValueError("HF token not found — set it in your secrets.")

In [4]:
# ── GPU check ─────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not detected. Set runtime to GPU.")

GPU_NAME       = torch.cuda.get_device_name(0)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
IS_H100        = "h100" in GPU_NAME.lower() or GPU_CAPABILITY[0] >= 9

print(f"Device : {GPU_NAME}  (cc {GPU_CAPABILITY[0]}.{GPU_CAPABILITY[1]})")
print(f"H100 profile : {IS_H100}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

Device : Tesla T4  (cc 7.5)
H100 profile : False


In [5]:
# ── Config ────────────────────────────────────────────────────────────────────
BASE_MODEL_ID  = "google/medgemma-1.5-4b-it"
ADAPTER_HUB_ID = "gabrielbuzzi/medgemma-aci-lora"   # your uploaded adapters

MAX_SEQ_LENGTH        = 4096
MAX_NEW_TOKENS        = 1024
GEN_REPETITION_PENALTY = 1.15
# NOTE: no_repeat_ngram_size is intentionally omitted — it blocks clinical
# terms that legitimately repeat between the dialogue and the note.

TEST_SUBSET_SIZE = 256   # set to 0 to run the full test set

VERBOSE_EVAL         = True
VERBOSE_EVAL_SAMPLES = 3

SYSTEM_PROMPT = """
You are an ambient clinical documentation assistant. Given a verbatim
transcript of a doctor-patient encounter, generate a structured
SOAP-style outpatient progress note.

### Guidelines
- Use clear ALL-CAPS section headings appropriate to the content
  (e.g., CHIEF COMPLAINT, HISTORY OF PRESENT ILLNESS, MEDICAL HISTORY,
  REVIEW OF SYSTEMS, VITALS, PHYSICAL EXAM, RESULTS, ASSESSMENT AND PLAN,
  INSTRUCTIONS).
- Write in a professional clinical style using third-person narrative prose.
- Include only sections for which information is present in the dialogue.
- Do not fabricate any information not explicitly stated in the dialogue.
- Preserve all medication names, doses, and frequencies exactly as spoken.
- The input transcript contains no speaker labels — infer clinical content
  from context.
""".strip()

In [6]:
# ── Prompt helpers (identical to training notebook) ───────────────────────────

def clean_dialogue(text: str) -> str:
    text = re.sub(r"\[doctor\]",  "", text, flags=re.IGNORECASE)
    text = re.sub(r"\[patient\]", "", text, flags=re.IGNORECASE)
    text = text.replace("\\n", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()


def build_dialogue_user_content(dialogue: str) -> str:
    return f"<dialogue>\n{dialogue}\n</dialogue>"


def wrap_note(note: str) -> str:
    return f"<note>\n{note.strip()}\n</note>"


def build_messages(dialogue: str, note: str | None = None) -> list[dict]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_dialogue_user_content(dialogue)},
    ]
    if note is not None:
        messages.append({"role": "assistant", "content": wrap_note(note)})
    return messages


def extract_prompt_messages(messages: list[dict]) -> list[dict]:
    return [m for m in messages if m["role"] in {"system", "user"}]


def build_prompt_text(tokenizer_obj, messages: list[dict]) -> str:
    chat_template = getattr(tokenizer_obj, "chat_template", None)
    if chat_template:
        return tokenizer_obj.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    chunks = []
    for m in messages:
        chunks.append(f"{m['role'].upper()}:\n{m['content']}")
    return "\n\n".join(chunks) + "\n\nASSISTANT:\n"


def prepare_example(example: dict[str, Any], noise_prob: float = 0.0) -> dict[str, Any]:
    example["input_dialogue"] = clean_dialogue(example["dialogue"])
    return example

In [7]:
# ── Load tokenizer ─────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
print(f"Chat template present: {bool(getattr(tokenizer, 'chat_template', None))}")

config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

Chat template present: True


In [8]:
# ── Load base model + merge LoRA adapters ─────────────────────────────────────
print("Loading base model...")

if IS_H100:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
        device_map={"": 0},
    )
else:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )

print("Attaching LoRA adapters from HuggingFace...")
model = PeftModel.from_pretrained(base_model, ADAPTER_HUB_ID)

# Critical fixes that were missing in the training notebook:
model.config.use_cache = True          # was left False after training
# Preserve the model's native eos list (e.g. [1, 107] for Gemma3)
native_eos = model.generation_config.eos_token_id
if native_eos is None:
    eos_ids = [tokenizer.eos_token_id]
elif isinstance(native_eos, int):
    eos_ids = [native_eos]
else:
    eos_ids = list(native_eos)
model.generation_config.eos_token_id = eos_ids
model.eval()

print("Model ready.")
print(f"EOS token ids : {eos_ids}")

Loading base model...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Attaching LoRA adapters from HuggingFace...


adapter_config.json:   0%|          | 0.00/977 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/12.9M [00:00<?, ?B/s]

Model ready.
EOS token ids : [1, 106]


In [9]:
# ── Load & prepare test dataset ───────────────────────────────────────────────
dataset = load_dataset("mkieffer/ACI-Bench", "aci")
test_ds  = concatenate_datasets([dataset["test1"], dataset["test2"], dataset["test3"]])
test_ds  = test_ds.map(prepare_example)

if TEST_SUBSET_SIZE > 0:
    test_ds = test_ds.select(range(min(TEST_SUBSET_SIZE, len(test_ds))))

print(f"Test samples : {len(test_ds)}")

README.md: 0.00B [00:00, ?B/s]

aci/train.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

aci/valid.parquet:   0%|          | 0.00/59.8k [00:00<?, ?B/s]

aci/test1.parquet:   0%|          | 0.00/98.3k [00:00<?, ?B/s]

aci/test2.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

aci/test3.parquet:   0%|          | 0.00/115k [00:00<?, ?B/s]

... (3 outputs left out)

Generating test2 split:   0%|          | 0/22 [00:00<?, ? examples/s]

Generating test3 split:   0%|          | 0/22 [00:00<?, ? examples/s]

Map:   0%|          | 0/66 [00:00<?, ? examples/s]

Test samples : 66


In [10]:
# ── Generation ────────────────────────────────────────────────────────────────

def generate_predictions(eval_dataset, inference_model, tokenizer_obj) -> list[str]:
    predictions = []
    for i, example in enumerate(eval_dataset, 1):
        print(f"Sample {i}/{len(eval_dataset)}", end="\r")

        prompt_messages = extract_prompt_messages(
            build_messages(dialogue=example["input_dialogue"])
        )
        prompt_text = build_prompt_text(tokenizer_obj, prompt_messages)

        inputs = tokenizer_obj(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            add_special_tokens=False,
        ).to(inference_model.device)

        with torch.no_grad():
            output_ids = inference_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                repetition_penalty=GEN_REPETITION_PENALTY,
                # no_repeat_ngram_size intentionally omitted — see config comment
                pad_token_id=tokenizer_obj.eos_token_id,
                eos_token_id=eos_ids,
            )

        generated_ids  = output_ids[0][inputs["input_ids"].shape[1]:]
        generated_text = tokenizer_obj.decode(generated_ids, skip_special_tokens=True).strip()

        if VERBOSE_EVAL and i <= VERBOSE_EVAL_SAMPLES:
            print(f"\n--- Sample {i} ---")
            print(f"EXPECTED:\n{example['note']}")
            print(f"\nGENERATED:\n{generated_text}")
            print("-" * 60)

        predictions.append(generated_text)

    print(f"\nDone. Generated {len(predictions)} predictions.")
    return predictions


predictions = generate_predictions(test_ds, model, tokenizer)

Sample 1/66
--- Sample 1 ---
EXPECTED:
CHIEF COMPLAINT

Shortness of breath.

HISTORY OF PRESENT ILLNESS
Charles Sullivan is a pleasant 45-year-old male who presents to the clinic today for the evaluation of shortness of breath. The onset of his symptoms began 2 years ago. He has noticed that his breathing is not what it used to be. He describes an increased need to catch his breath. The patient also reports he gets "mid-shortness of breath" without reason. He has been seen at an urgent care where he received a nebulizer treatment and was prescribed an inhaler, which he uses when his symptoms are present. The patient states his symptoms are aggravated by living with a cat in his house and are present even when the cat is not around or is outside. He has experienced epiphora with rheum present that he attributes to allergies. He has had diarrhea after consuming large amounts of alcohol. He denies any symptoms at this time including shortness of breath, rash, nausea, vomiting, and lip or

KeyboardInterrupt: 

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────

rouge_metric    = evaluate.load("rouge")
bleu_metric     = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")


def strip_note_wrapper(text: str) -> str:
    """Remove <note>...</note> wrapper tags from generated text."""
    text = text.strip()
    text = re.sub(r"^\s*<note>\s*",  "", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"\s*</note>\s*$", "", text, flags=re.IGNORECASE | re.DOTALL)
    return text.strip()


def compute_cross_entropy_and_perplexity(
    eval_dataset, eval_model, eval_tokenizer
) -> tuple[float, float]:
    """Reference-conditioned CE/PPL with prompt tokens masked out."""
    eval_model.eval()
    total_loss, total_tokens = 0.0, 0

    with torch.no_grad():
        for example in eval_dataset:
            prompt_messages = extract_prompt_messages(
                build_messages(dialogue=example["input_dialogue"])
            )
            prompt_text    = build_prompt_text(eval_tokenizer, prompt_messages)
            reference_text = wrap_note(example["note"])
            full_text      = f"{prompt_text}{reference_text}{eval_tokenizer.eos_token}"

            inputs = eval_tokenizer(
                full_text,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
                add_special_tokens=False,
            ).to(eval_model.device)

            prompt_len = min(
                len(eval_tokenizer(
                    prompt_text,
                    truncation=True,
                    max_length=MAX_SEQ_LENGTH,
                    add_special_tokens=False,
                )["input_ids"]),
                inputs["input_ids"].shape[1],
            )

            labels = inputs["input_ids"].clone()
            labels[:, :prompt_len] = -100

            loss        = eval_model(**inputs, labels=labels).loss
            token_count = int((labels != -100).sum().item())
            total_loss  += float(loss.item() * token_count)
            total_tokens += token_count

    avg_ce = total_loss / max(total_tokens, 1)
    ppl    = float(math.exp(avg_ce)) if avg_ce < 20 else float("inf")
    return avg_ce, ppl


def compute_text_metrics(eval_dataset, preds: list[str], eval_model, eval_tokenizer) -> dict:
    cleaned_preds = [strip_note_wrapper(p) for p in preds]

    # Guard against any remaining empty predictions (prevents BLEU division-by-zero)
    empty = sum(1 for p in cleaned_preds if not p.strip())
    if empty:
        print(f"WARNING: {empty} empty predictions detected — check generation output.")
    cleaned_preds = [p if p.strip() else " " for p in cleaned_preds]

    references = eval_dataset["note"]

    rouge     = rouge_metric.compute(predictions=cleaned_preds, references=references)
    bleu      = bleu_metric.compute(predictions=cleaned_preds, references=[[r] for r in references])
    bertscore = bertscore_metric.compute(predictions=cleaned_preds, references=references, lang="en")
    ce, ppl   = compute_cross_entropy_and_perplexity(eval_dataset, eval_model, eval_tokenizer)

    return {
        "rouge1"      : float(rouge["rouge1"]),
        "rouge2"      : float(rouge["rouge2"]),
        "rougeL"      : float(rouge["rougeL"]),
        "bleu"        : float(bleu["bleu"]),
        "bertscore_f1": float(np.mean(bertscore["f1"])),
        "cross_entropy": ce,
        "perplexity"  : ppl,
    }


print("Computing metrics...")
metrics = compute_text_metrics(test_ds, predictions, model, tokenizer)

print("\n===== Evaluation Metrics =====")
for k, v in metrics.items():
    print(f"  {k:<20} {v:.4f}")